# Stationary baseline analysis

Use this notebook to inspect sensor noise, DC removal, RMS, and window statistics while the monitored surface is stationary. These measurements establish the baseline used when selecting the idle threshold.

> The notebook is exploratory. Reusable feature calculations belong in `src/machine_sentinel/features.py`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('../../data/raw/stationary.csv')

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df['time_s'] = (df['timestamp_us'] - df['timestamp_us'][0]) / 1000000

In [ ]:
df.head()

In [ ]:
acceleration = df['az']
time = df['time_s']

In [ ]:
plt.plot(time, acceleration, label="Accl vs Time", color='b', lw=2)
plt.title('Basic Acceleration vs Time Plot')
plt.xlabel('Time (s)')
plt.ylabel('Raw Z Acceleration (counts)')

In [ ]:
mean = np.mean(acceleration)
standard_deviation = np.std(acceleration)
peak_to_peak = np.ptp(acceleration)
rms = np.sqrt(np.mean(np.square(acceleration)))

In [ ]:
print(f"Mean: {mean}")  
print(f"Standard Deviation: {standard_deviation}")
print(f"Peak-to-Peak: {peak_to_peak}")
print(f"RMS: {rms}")

In [ ]:
z_centered = acceleration - mean

In [ ]:
plt.plot(time, z_centered, label="Accl vs Time", color='b', lw=2)
plt.title('Z-Centered Acceleration vs Time Plot')
plt.xlabel('Time (s)')
plt.ylabel('Z-Centered Acceleration')

In [ ]:
rms_z_centered = np.sqrt(np.mean(np.square(z_centered)))
print(f"RMS of Z-Centered Acceleration: {rms_z_centered}")
print(f"Standard Deviation of Z-Centered Acceleration: {np.std(z_centered)}")
print(f"Peak-to-Peak of Z-Centered Acceleration: {np.ptp(z_centered)}")
print(f"Mean of Z-Centered Acceleration: {np.mean(z_centered)}")

In [ ]:
WINDOW_SIZE = 200  
data = []  

for start in range(0, len(df), WINDOW_SIZE):     
    end = min(start + WINDOW_SIZE, len(df))      
    window = df.iloc[start:end]     
    
    # Convert to g's securely
    z_g = window['az'] / 4096.0  
    
    # Calculate statistics
    z_centered_window = z_g - z_g.mean()     
    rms_window = np.sqrt(np.mean(np.square(z_centered_window)))     
    std_window = z_centered_window.std()     
    
    # Fixed: Use max - min instead of np.ptp to ensure safe Pandas compatibility
    ptp_window = z_g.max() - z_g.min()      

    data.append({         
        'start_idx': start,
        'end_idx': end,
        'rms': rms_window,         
        'std': std_window,         
        'ptp': ptp_window,         
        'condition': 'stationary'     
    })  

window_stats = pd.DataFrame(data)

In [ ]:
window_stats.head()